# (Ab initio) Prediction of Intrinsically Disordered Regions in Proteins

## Uvod


**TODO**: Ovde treba uvod....

https://cs229.stanford.edu/proj2016/report/Attia-EnsemblePredictionOfIntrinsicallyDisorderedRegionsInProteins-report.pdf

### Terminologija


**Protein** je lanac aminokiselina. Sve što je bitno u kontekstu ovog projekta je da je on opisan niskom velikih slova engleske abecede. Na engleskom *protein*.

**Aminokiselina** je osnovni gradivni blok proteina. U niski koja opisuje protein, one su individualni karakteri. Postoje 20 standardnih aminokiselina -- one će u projektu biti označene velikim latiničnim slovima iz skupa `ACDEFGHIKLMNPQRSTVWY`. Pored njih, mogu se pojaviti i nestandardne aminokiseline, koje ćemo označavati `UNK` vrednostima. Na engleskom *residue*.

**Oblast** (ili **regija**) je povezana komponenta unutar proteina koja opisuje strukturnu informaciju. Izvori skupa podataka (navedeni dole) ovu informaciju dele binarno na *uredjen* i *neuredjen*, mada *neuredjen* opisuje razna moguca stanja. Preciznije, *uredjen* opisuje oblasti koje su se savile i imaju sekundarnu i tercijarnu strukturu. Klasa *neuredjen* obuhvata sve oblasti od onih potpuno neuređenih do onih koje imaju sekundarnu, ali ne i tercijarnu strukturu. Na engleskom *region*.

**TODO** citirati ovo...


### Skupovi podataka

Postoje dva glavna izvora skupa podataka: **DisProt** (https://disprot.org/) i **MobiDB** (https://mobidb.org/). Oba održava BiocomputingUP Lab Unverziteta u Padovi. 

DisProt skup sastavljen je od ručno proverenih podataka. Zbog toga je vrlo precizan, ali i mali -- sadrži oko 3500 proteina. Takođe, neke proteinske regije nisu testirane, pa u skupu podataka postoje i regije za koje je klasifikacija nepoznata.

MobiDB je, sa druge strane, sastavljen od velikog broja podataka. Nastoji da za svaku regiju svakog proteina iz **UniProt** baze podataka (https://www.uniprot.org/) izvrši predviđanje uređenosti. Predviđanja u MobiDB dele se na četiri kategorije, u redosledu pouzdanosti:

1. Curated — tačne vrednosti iz DisProt.
2. Derived — vrednosti indirektno dobijene eksperimentima rendgenske kristalografije i nuklearne magnetne rezonancije. Iz ova dva eksperimenta možemo, sa dovoljno jakom uverenošću, utvrditi da li je neka oblast uređena ili ne. Zbog toga nam služi kao pouzdana dopuna na podatake prve kategorije.
3. Homology — rezultati predviđeni na osnovu tačnih rezultata sličnih sekvenci.
4. Prediction — rezultati dobijeni raznim predviđačkim modelima.


U ovom projektu pokušaćemo da zaključke izvedemo iz istinitih *(ground truth)* vrednosti, a ne onih predviđenih od strane drugih modela. Prema tome, koristićemo samo MobiDB podatke prve i druge kategorije.


.

.

.


**TODO** urediti citiranje




DisProt in 2026: enhancing intrinsically disordered proteins accessibility, deposition, and annotation
Nugnes MV, Bouhraoua KEA, Zoubiri M, Pancsa R, Fichó E, DisProt Consortium, Tompa P, Piovesan D, Tosatto SCE and Aspromonte MC (2025) Nucleic Acids Research, Database Issue.



## Importi

In [30]:
import json
import time
import urllib.request
import urllib.parse
import csv

## Generisanje skupa podataka iz DisProt

Disprot skup podataka nalazi se u `data/raw_data/DisProt_current_consensus.json`. Dobijen je preuzimanjem trenutne verzije (`2026_06`) sa zvaničnog sajta. (https://disprot.org/api/v2/download?format=json&release=current&get_consensus=)

In [2]:
DISPROT_PATH = 'data/raw_data/DisProt_current_consensus.json'

In [3]:
class Protein_data:
    accession: str
    sequence: str
    labels: list
    
    def __init__(self, _accession, _sequence, _labels):
        self.accession = _accession
        self.sequence = _sequence
        self.labels = _labels
    

In [4]:
def parse_disprot_protein_entry(raw_protein_entry):
    
    _accession = raw_protein_entry.get('acc')
    assert(_accession is not None)
    
    _sequence = raw_protein_entry.get('sequence')
    assert(_sequence is not None)
    
    # 'Structural state nam aminokiseline u tranziciji klasifikuje kao neuredjene, kao sto smo i rekli da zelimo
    consensus = raw_protein_entry.get('disprot_consensus').get('Structural state')
    assert(consensus is not None)
    
        
    # Na pocetku su sve aminokisline -1, tj. neodredjene strukture
    _labels = []
    for i in range(len(_sequence)):
        _labels.append(-5)
    
    for region in consensus:
        # disprot indeksira od jedinice
        l = int(region['start']) - 1
        r = int(region['end'])
        
        l = max(l, 0)
        r = min(r, len(_labels))
        
        tip = region['type'].upper()
        
        lb = 0
        if tip == 'D':
            lb = 1
        elif tip == 'O':
            lb = 0
        else:
            continue
        
        for i in range(l, r):
            # proveramo da ne postoje konflikti u klasifikaciji uredjenja
            assert(_labels[i] == lb or _labels[i] == -5)
            _labels[i] = lb
        
        #print(_labels)
        
    return Protein_data(_accession, _sequence, _labels)
        

def parse_disprot(path):
    with open(path) as disprot:
        disprot_raw_data = json.load(disprot)
    disprot_raw_data = disprot_raw_data['data']

    proteins = []
    
    for raw_protein_entry in disprot_raw_data:
        protein_data = parse_disprot_protein_entry(raw_protein_entry)
        proteins.append(protein_data)
    
    print("Protein data successfully parsed")
    return proteins

In [5]:
proteins = parse_disprot(DISPROT_PATH)

Protein data successfully parsed


In [6]:
print(len(proteins))

3337


In [7]:
def count_values(proteins):
    ordered_count = 0
    disordered_count = 0
    undetermined_count = 0
    
    for protein in proteins:
        for l in protein.labels:
            if l == 0:
                ordered_count += 1
            elif l == 1:
                disordered_count += 1
            else:
                undetermined_count += 1
                
    print("Ordered residues:", ordered_count)
    print("Disordered residues:", disordered_count)
    print("Undetermined residues:", undetermined_count)

In [8]:
count_values(proteins)

Ordered residues: 0
Disordered residues: 336042
Undetermined residues: 1615606


DisProt uopste ne sadrži informacije o uređenim oblastima. Dalje možemo postupiti na dva načina:

1. Pretpostaviti da su sve neodređene aminokiseline zapravo deo neke uređene oblasti
2. Potvrdu o uređenosti naći iz MobiDB

Prvi metod je vrlo rizičan: veliki broj neutvrđenih oblasti zaista nije testiran, pa nije precizno da ih smatramo za uređene. Zato se opredeljujemo za drugi pristup.

Svakom proteinu pamtili smo `accession` vrednost. Ona predstavlja identifikator proteina u UniProt bazi podataka i možemo je koristiti kako bi smo iz MobiDB dobili informacije o pojedinačnim proteinima.

In [15]:
MOBIDB_PATH = 'data/mobidb.json'

In [13]:
def download_mobidb(proteins):
    accession_ids = [p.accession for p in proteins]
    
    mobidb_entries = []
    
    batch_size = 500
    for t in range(len(accession_ids)):
        l = t*batch_size
        r = t*batch_size + batch_size
        
        if l >= len(accession_ids):
            break
        
        r = min(r, len(accession_ids))
        
        acc_list = ",".join(accession_ids[l:r])
        url = "https://mobidb.org/api/download_page?" + urllib.parse.urlencode({"limit": 1000, "acc": acc_list})
        print("Requesting", r - l, "accessions")
        
        with urllib.request.urlopen(url, timeout=120) as r:
            raw_data = r.read().decode()
        
        data = json.loads(raw_data)
        data = data["data"]
        
        print("Got", len(data), "entries")
        
        # Ume da blokira ako ne sacekamo
        time.sleep(1)
        
        mobidb_entries.extend(data)
    
    json.dump({"data": mobidb_entries}, open(MOBIDB_PATH, "w"))
    print("Saved", len(mobidb_entries), "entries to ", MOBIDB_PATH)

In [14]:
download_mobidb(proteins)

Requesting 500 accessions
Got 499 entries
Requesting 500 accessions
Got 500 entries
Requesting 500 accessions
Got 500 entries
Requesting 500 accessions
Got 500 entries
Requesting 500 accessions
Got 500 entries
Requesting 500 accessions
Got 500 entries
Requesting 337 accessions
Got 332 entries
Saved 3331 entries to mobidb.json


Neke datoteke fale. Takvih ima samo 6, pa nije preveliki problem, ali svakako možemo da vidimo o čemu se radi.

In [24]:
def print_missing_accs(requested_accs):
    requested_accs_set = set(requested_accs)
    returned_accs = []
    with open(MOBIDB_PATH) as mobidb_file:
        mobidb_file = json.load(mobidb_file)
    mobidb_file = mobidb_file['data']
    
    for entry in mobidb_file:
        returned_accs.append(entry['acc'])
    
    returned_accs_set = set(returned_accs)

    missing_accs = requested_accs_set - returned_accs_set
    
    print("Missing", len(missing_accs), "accs")
    for acc in missing_accs:
        print(acc)

In [26]:
print_missing_accs([p.accession for p in proteins])

Missing 6 accs
A0A9R1FWH6
Q62871-3
P09651-2
A0ABQ8E5R8
A0ABF7SXN4
O43236-1


Za ove proteine koristićemo samo vrednosti iz DisProt.

In [98]:
def merge_labels(protein_labels, mobidb_labels):
    final_labels = []
    assert(len(protein_labels) == len(mobidb_labels))
    for i in range(len(protein_labels)):
        if protein_labels[i] == 1:
            # sigurno 1
            final_labels.append(1)
        elif mobidb_labels[i] == 1:
            # pretpostavicemo da ako mobidb dobija 1, to zaista jeste neuredjenje
            final_labels.append(1)
        elif mobidb_labels[i] == 0:
            # pretpostavljamo da ako rucno nije dobijeno nista, a mobidb dobija 0, to zaista jeste uredjenje 
            final_labels.append(0)
        else:
            # inace ne mozemo nista novo da pretpostavimo
            final_labels.append(protein_labels[i])
    return final_labels
            
def get_mobidb_labels_from_entry(entry, sequence_length):
    labels = []
    for i in range(sequence_length):
        labels.append(-5)
        
    for key, value in entry.items():
        if(key.startswith("derived-observed")):
            # ovo su novodobijene uredjene vrednosti iz eksperimenata
            for region in value["regions"]:
                l = region[0] - 1
                r = region[1]
                
                l = max(0, l)
                r = min(r, sequence_length)
                
                for i in range(l, r):
                    labels[i] = 0
    
    for key, value in entry.items():
        if(key.startswith("derived-missing_residues") or key.startswith("curated-disorder")):
            # novodobijene neuredjene vrednosti iz eksperimenata, kao i one iz DisProt
            for region in value["regions"]:
                l = region[0] - 1
                r = region[1]
                
                l = max(0, l)
                r = min(r, sequence_length)
                
                for i in range(l, r):
                    labels[i] = 1
                    
    return labels

def merge_proteins(proteins, mobidb_data):
    merged_proteins = []
    for protein in proteins:
        found_acc_match = False
        for entry in mobidb_data:
            if(protein.accession == entry['acc']):
                # TODO pogledati zasto se desava length mismatch
                #length = entry.get('length')
                #if length is None:
                
                length = len(protein.labels)
                mobidb_labels = get_mobidb_labels_from_entry(entry, length)

                merged_labels = merge_labels(protein.labels, mobidb_labels)

                merged_proteins.append(Protein_data(protein.accession, protein.sequence, merged_labels))
                
                found_acc_match = True
                break
        if found_acc_match == False:
            merged_proteins.append(Protein_data(protein.accession, protein.sequence, protein.labels))
    
    return merged_proteins

In [99]:
with open(MOBIDB_PATH) as mobidb_file:
    mobidb_file = json.load(mobidb_file)
mobidb_data = mobidb_file['data']
    
merged_proteins = merge_proteins(proteins, mobidb_data)

In [100]:
print(len(merged_proteins))

3337


In [101]:
count_values(merged_proteins)

Ordered residues: 758336
Disordered residues: 576684
Undetermined residues: 616628


Dobili smo 57:43 podelu, što je u skladu sa 60:40 podelom koju je rad spomenuo.

Da nismo objedinili podatke sa MobiDB, već pretpostavili da su sve neodređene vrednosti iz DisProt uređene, dobili bismo 83:17 podelu i rezultati bi bili dosta manje precizni.

Čuvamo dobijene podatke.

In [105]:
headers = ["accession", "sequence", "labels"]

with open("data/protein_data.csv", "w") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=headers)
    
    writer.writeheader()
    
    for protein in merged_proteins:
        writer.writerow(protein.__dict__)